# CGR Network Pruning, LOO Ablation & FDA Replacement Screen (Turnkey Colab)

Self-contained Google Colab pipeline querying **ChEMBL**, **Open Targets**, and **STRING** live to run affinity-weighted leave-one-out (LOO) ablation across **MDD, BIP, SCZ, and OCD** and screen for FDA-approved (Phase 4) single-molecule replacements.

**What's pre-wired & fixed vs. the draft script:**
1. **Zero uploads needed:** All 121 genes extracted from `cgr.grok.me` (`106` GMT-mapped CGR anchors + `15` Pharm Anchors) are embedded directly.
2. **Exact ChEMBL parent-molecule lookup:** Uses `pref_name__iexact` before fuzzy search (avoiding salt/derivative mismatches) and reuses verified human single-protein target ChEMBL IDs during the FDA screen.
3. **Dual sensitivity mode (`INCLUDE_METABOLIC`):** Runs both **binding-only** (ChEMBL $\text{pChEMBL} \ge 5.5$) and **full-stack with metabolic/structural precursors** (`L-glutamine`, `NAC`, `NMN`, `Citicoline`, `DHA`, `Piracetam`) side by side.
4. **Automatic `.zip` export:** Downloads all ablation tables, ChEMBL profiles, FDA replacement candidates (with network proximity scores), and a CGR-Lite pruning summary.


In [13]:
# @title 1. Configuration, Dependencies, Disk Cache & Embedded CGR Anchors (121 genes)
import os, io, time, json, zipfile, hashlib
from pathlib import Path
import requests
import numpy as np
import pandas as pd
import networkx as nx

os.makedirs("data", exist_ok=True)
os.makedirs("data/cache", exist_ok=True)
os.makedirs("results", exist_ok=True)

# ---- Core thresholds -------------------------------------------------------
PCHEMBL_CUTOFF         = 5.5          # @param {type:"number"}
STRING_SCORE_CUTOFF    = 700          # @param {type:"integer"}
CYP2D6_INHIBITOR       = "bupropion"  # @param ["bupropion", "paroxetine", "duloxetine", "quinidine"]
METABOLIC_PSEUDO_PCHEMBL = 6.0        # @param {type:"number"}

# ---- NEW: sensitivity, PK gating & null-model controls ---------------------
PSEUDO_PCHEMBL_GRID = [5.5, 6.0, 6.5] # @param  rank-stability sweep for curated pseudo-affinity
PK_DXM_PENALTY      = 1.0             # @param {type:"number"}  pChEMBL attenuation applied to DXM when CYP2D6i is dropped
N_PERM              = 200             # @param {type:"integer"} degree-matched permutations per component
RANDOM_SEED         = 1234            # @param {type:"integer"}
FDA_ACT_PCHEMBL_MIN = 6.5             # @param {type:"number"}  potency floor for FDA activity-tier screen

# ---------------------------------------------------------------------------
# Generic on-disk JSON cache so iterative sensitivity/null runs are instant.
# ---------------------------------------------------------------------------
def _cache_path(key: str) -> Path:
    h = hashlib.md5(key.encode("utf-8")).hexdigest()
    return Path("data/cache") / f"{h}.json"

def cached_json(key, fetch_fn):
    """Return cached JSON for `key`, else call fetch_fn(), store, return."""
    p = _cache_path(key)
    if p.exists():
        try:
            return json.loads(p.read_text(encoding="utf-8"))
        except Exception:
            pass
    data = fetch_fn()
    try:
        p.write_text(json.dumps(data), encoding="utf-8")
    except Exception:
        pass
    return data

ANCHORS_CSV = """symbol,ensembl_id,module,is_pharm_anchor,functional_family,pathway_sets
GRIN1,,LTP_init,False,Glutamate Receptors & Homeostasis,hsa04720;hsa04724;GO:0035235;GO:0007215
GRIN2A,,LTP_init,False,Glutamate Receptors & Homeostasis,hsa04720;hsa04724;GO:0060291;GO:0035235;GO:0007215
GRIN2B,,LTP_init,False,Glutamate Receptors & Homeostasis,hsa04720;hsa04724;GO:0060291;GO:0035235;GO:0007215
GRIN2C,,LTP_init,False,Glutamate Receptors & Homeostasis,hsa04720;hsa04724;GO:0060291;GO:0035235;GO:0007215
GRIN2D,,LTP_init,False,Glutamate Receptors & Homeostasis,hsa04720;hsa04724;GO:0060291;GO:0035235;GO:0007215
GRIA1,,LTP_init,False,Glutamate Receptors & Homeostasis,hsa04720;hsa04724
GRIA2,,LTP_init,False,Glutamate Receptors & Homeostasis,hsa04720;hsa04724;GO:0035235;GO:0007215
GRIA3,,LTP_init,False,Glutamate Receptors & Homeostasis,hsa04724;GO:0007215
GRIA4,,LTP_init,False,Glutamate Receptors & Homeostasis,hsa04724;GO:0007215
GRM2,,LTP_init,False,Glutamate Receptors & Homeostasis,hsa04724;GO:0007215
GRM3,,LTP_init,False,Glutamate Receptors & Homeostasis,hsa04724;GO:0007215
SLC1A1,,LTP_init,False,Glutamate-Glutamine Cycle & Vesicular Release,hsa04724
SLC1A2,,LTP_init,False,Glutamate-Glutamine Cycle & Vesicular Release,hsa04724
SLC1A3,,LTP_init,False,Glutamate-Glutamine Cycle & Vesicular Release,hsa04724
SLC1A6,,LTP_init,False,Glutamate-Glutamine Cycle & Vesicular Release,hsa04724
SLC1A7,,LTP_init,False,Glutamate-Glutamine Cycle & Vesicular Release,hsa04724
GLUL,,LTP_init,False,Glutamate-Glutamine Cycle & Vesicular Release,hsa04724
SLC38A1,,LTP_init,False,Glutamate-Glutamine Cycle & Vesicular Release,hsa04724
SLC38A2,,LTP_init,False,Glutamate-Glutamine Cycle & Vesicular Release,hsa04724
SLC38A3,,LTP_init,False,Glutamate-Glutamine Cycle & Vesicular Release,hsa04724
GLS,,LTP_init,False,Glutamate-Glutamine Cycle & Vesicular Release,hsa04724
GLS2,,LTP_init,False,Glutamate-Glutamine Cycle & Vesicular Release,hsa04724
SLC17A6,,LTP_init,False,Glutamate-Glutamine Cycle & Vesicular Release,hsa04724
SLC17A7,,LTP_init,False,Glutamate-Glutamine Cycle & Vesicular Release,hsa04724
SLC17A8,,LTP_init,False,Glutamate-Glutamine Cycle & Vesicular Release,hsa04724
VAMP2,,LTP_init,False,Glutamate-Glutamine Cycle & Vesicular Release,GO:0060291
STX4,,LTP_init,False,Glutamate-Glutamine Cycle & Vesicular Release,GO:0060291
CACNA1C,,LTP_init,False,"Ca2+, cAMP & ERK-CREB Consolidation",hsa04720;hsa04724
CALM1,,LTP_init;TrkB_mTOR,False,"Ca2+, cAMP & ERK-CREB Consolidation",hsa04720;hsa04722
CALM2,,LTP_init;TrkB_mTOR,False,"Ca2+, cAMP & ERK-CREB Consolidation",hsa04720;hsa04722
CALM3,,LTP_init;TrkB_mTOR,False,"Ca2+, cAMP & ERK-CREB Consolidation",hsa04720;hsa04722
CAMK2A,,LTP_init;TrkB_mTOR,False,"Ca2+, cAMP & ERK-CREB Consolidation",hsa04720;hsa04722
CAMK2B,,LTP_init;TrkB_mTOR;Actin_spine,False,"Ca2+, cAMP & ERK-CREB Consolidation",hsa04720;hsa04722;GO:0061001
ADCY1,,LTP_init,False,"Ca2+, cAMP & ERK-CREB Consolidation",hsa04720;hsa04724
ADCY8,,LTP_init,False,"Ca2+, cAMP & ERK-CREB Consolidation",hsa04720;hsa04724
PRKACA,,LTP_init,False,"Ca2+, cAMP & ERK-CREB Consolidation",hsa04720;hsa04724
PRKACB,,LTP_init,False,"Ca2+, cAMP & ERK-CREB Consolidation",hsa04720;hsa04724
PRKACG,,LTP_init,False,"Ca2+, cAMP & ERK-CREB Consolidation",hsa04720;hsa04724
RAPGEF3,,LTP_init,False,"Ca2+, cAMP & ERK-CREB Consolidation",hsa04720
RAP1A,,LTP_init;TrkB_mTOR,False,"Ca2+, cAMP & ERK-CREB Consolidation",hsa04720;hsa04722
RAP1B,,LTP_init;TrkB_mTOR,False,"Ca2+, cAMP & ERK-CREB Consolidation",hsa04720;hsa04722
HRAS,,LTP_init;TrkB_mTOR;Actin_spine,False,"Ca2+, cAMP & ERK-CREB Consolidation",hsa04720;hsa04722;hsa04150;hsa04151;hsa04810
BRAF,,LTP_init;TrkB_mTOR;Actin_spine,False,"Ca2+, cAMP & ERK-CREB Consolidation",hsa04720;hsa04722;hsa04150;hsa04810
RAF1,,LTP_init;TrkB_mTOR;Actin_spine,False,"Ca2+, cAMP & ERK-CREB Consolidation",hsa04720;hsa04722;hsa04150;hsa04151;hsa04810
MAP2K1,,LTP_init;TrkB_mTOR;Actin_spine,False,"Ca2+, cAMP & ERK-CREB Consolidation",hsa04720;hsa04722;hsa04150;hsa04151;hsa04810
MAP2K2,,LTP_init;TrkB_mTOR;Actin_spine,False,"Ca2+, cAMP & ERK-CREB Consolidation",hsa04720;hsa04722;hsa04150;hsa04151;hsa04810
MAPK1,,LTP_init;TrkB_mTOR;Actin_spine,False,"Ca2+, cAMP & ERK-CREB Consolidation",hsa04720;hsa04724;hsa04722;hsa04150;hsa04151;hsa04810
MAPK3,,LTP_init;TrkB_mTOR;Actin_spine,False,"Ca2+, cAMP & ERK-CREB Consolidation",hsa04720;hsa04724;hsa04722;hsa04150;hsa04151;hsa04810
RPS6KA1,,LTP_init;TrkB_mTOR,False,"Ca2+, cAMP & ERK-CREB Consolidation",hsa04720;hsa04722;hsa04150
RPS6KA2,,LTP_init;TrkB_mTOR,False,"Ca2+, cAMP & ERK-CREB Consolidation",hsa04720;hsa04722;hsa04150
RPS6KA3,,LTP_init;TrkB_mTOR,False,"Ca2+, cAMP & ERK-CREB Consolidation",hsa04720;hsa04722;hsa04150
RPS6KA6,,LTP_init;TrkB_mTOR,False,"Ca2+, cAMP & ERK-CREB Consolidation",hsa04720;hsa04722;hsa04150
CREBBP,,LTP_init,False,"Ca2+, cAMP & ERK-CREB Consolidation",hsa04720
EP300,,LTP_init,False,"Ca2+, cAMP & ERK-CREB Consolidation",hsa04720
ATF4,,LTP_init;TrkB_mTOR,False,"Ca2+, cAMP & ERK-CREB Consolidation",hsa04720;hsa04722;hsa04151
BDNF,,TrkB_mTOR;Adhesion_PSD,False,"Neurotrophin, PI3K-Akt-mTOR & Bioenergetic Gating",hsa04722;hsa04151;GO:0007416
NTRK2,,TrkB_mTOR,False,"Neurotrophin, PI3K-Akt-mTOR & Bioenergetic Gating",hsa04722;hsa04151
PIK3CA,,TrkB_mTOR;Actin_spine,False,"Neurotrophin, PI3K-Akt-mTOR & Bioenergetic Gating",hsa04722;hsa04150;hsa04151;hsa04810
AKT1,,TrkB_mTOR,False,"Neurotrophin, PI3K-Akt-mTOR & Bioenergetic Gating",hsa04722;hsa04150;hsa04151
AKT2,,TrkB_mTOR,False,"Neurotrophin, PI3K-Akt-mTOR & Bioenergetic Gating",hsa04722;hsa04150;hsa04151
AKT3,,TrkB_mTOR,False,"Neurotrophin, PI3K-Akt-mTOR & Bioenergetic Gating",hsa04722;hsa04150;hsa04151
STK11,,TrkB_mTOR,False,"Neurotrophin, PI3K-Akt-mTOR & Bioenergetic Gating",hsa04150;hsa04151
PRKAA1,,TrkB_mTOR,False,"Neurotrophin, PI3K-Akt-mTOR & Bioenergetic Gating",hsa04150;hsa04151
PRKAA2,,TrkB_mTOR,False,"Neurotrophin, PI3K-Akt-mTOR & Bioenergetic Gating",hsa04150;hsa04151
TSC1,,TrkB_mTOR,False,"Neurotrophin, PI3K-Akt-mTOR & Bioenergetic Gating",hsa04150;hsa04151
TSC2,,TrkB_mTOR,False,"Neurotrophin, PI3K-Akt-mTOR & Bioenergetic Gating",hsa04150;hsa04151
RHEB,,TrkB_mTOR,False,"Neurotrophin, PI3K-Akt-mTOR & Bioenergetic Gating",hsa04150;hsa04151
MTOR,,TrkB_mTOR,False,"Neurotrophin, PI3K-Akt-mTOR & Bioenergetic Gating",hsa04150;hsa04151;GO:0031929
RPTOR,,TrkB_mTOR,False,"Neurotrophin, PI3K-Akt-mTOR & Bioenergetic Gating",hsa04150;hsa04151;GO:0031929
RPS6KB1,,TrkB_mTOR,False,"Neurotrophin, PI3K-Akt-mTOR & Bioenergetic Gating",hsa04150;hsa04151;GO:0031929
EIF4E,,TrkB_mTOR,False,"Neurotrophin, PI3K-Akt-mTOR & Bioenergetic Gating",hsa04150;hsa04151
EIF4EBP1,,TrkB_mTOR,False,"Neurotrophin, PI3K-Akt-mTOR & Bioenergetic Gating",hsa04150;hsa04151;GO:0031929
RAC1,,TrkB_mTOR;Actin_spine,False,"Actin Cytoskeleton, Membrane Lipids & Spine Remodeling",hsa04722;hsa04151;hsa04810;GO:0007015
CDC42,,TrkB_mTOR;Actin_spine,False,"Actin Cytoskeleton, Membrane Lipids & Spine Remodeling",hsa04722;hsa04810;GO:0007015;GO:0007266
PAK1,,Actin_spine,False,"Actin Cytoskeleton, Membrane Lipids & Spine Remodeling",hsa04810
PAK3,,Actin_spine,False,"Actin Cytoskeleton, Membrane Lipids & Spine Remodeling",hsa04810;GO:0060997
CFL1,,Actin_spine,False,"Actin Cytoskeleton, Membrane Lipids & Spine Remodeling",hsa04810;GO:0061001;GO:0007266
ACTB,,Actin_spine,False,"Actin Cytoskeleton, Membrane Lipids & Spine Remodeling",hsa04810
ACTG1,,Actin_spine,False,"Actin Cytoskeleton, Membrane Lipids & Spine Remodeling",hsa04810
ACTR2,,Actin_spine,False,"Actin Cytoskeleton, Membrane Lipids & Spine Remodeling",hsa04810
ACTR3,,Actin_spine,False,"Actin Cytoskeleton, Membrane Lipids & Spine Remodeling",hsa04810
WASL,,Actin_spine,False,"Actin Cytoskeleton, Membrane Lipids & Spine Remodeling",hsa04810;GO:0060997;GO:0007015
DBN1,,Actin_spine,False,"Actin Cytoskeleton, Membrane Lipids & Spine Remodeling",GO:0061001;GO:0007015
PLA2G4A,,LTP_init,False,"Actin Cytoskeleton, Membrane Lipids & Spine Remodeling",hsa04724
PLA2G4B,,LTP_init,False,"Actin Cytoskeleton, Membrane Lipids & Spine Remodeling",hsa04724
PLA2G4C,,LTP_init,False,"Actin Cytoskeleton, Membrane Lipids & Spine Remodeling",hsa04724
PLA2G4D,,LTP_init,False,"Actin Cytoskeleton, Membrane Lipids & Spine Remodeling",hsa04724
PLA2G4E,,LTP_init,False,"Actin Cytoskeleton, Membrane Lipids & Spine Remodeling",hsa04724
PLA2G4F,,LTP_init,False,"Actin Cytoskeleton, Membrane Lipids & Spine Remodeling",hsa04724
PLD1,,LTP_init,False,"Actin Cytoskeleton, Membrane Lipids & Spine Remodeling",hsa04724
PLD2,,LTP_init,False,"Actin Cytoskeleton, Membrane Lipids & Spine Remodeling",hsa04724
PIP5K1A,,Actin_spine,False,"Actin Cytoskeleton, Membrane Lipids & Spine Remodeling",hsa04810
PIP5K1B,,Actin_spine,False,"Actin Cytoskeleton, Membrane Lipids & Spine Remodeling",hsa04810
PIP5K1C,,Actin_spine,False,"Actin Cytoskeleton, Membrane Lipids & Spine Remodeling",hsa04810
DLG4,,LTP_init;TrkB_mTOR;Actin_spine;Adhesion_PSD,False,"PSD Scaffolds, Trans-Synaptic Adhesion & Senescence/Survival",hsa04724;GO:0060997
SHANK1,,LTP_init;Actin_spine,False,"PSD Scaffolds, Trans-Synaptic Adhesion & Senescence/Survival",hsa04724;GO:0060997
SHANK2,,LTP_init;Adhesion_PSD,False,"PSD Scaffolds, Trans-Synaptic Adhesion & Senescence/Survival",hsa04724;GO:0060291;GO:0007416
SHANK3,,LTP_init;TrkB_mTOR;Actin_spine;Adhesion_PSD,False,"PSD Scaffolds, Trans-Synaptic Adhesion & Senescence/Survival",hsa04724;GO:0060997;GO:0061001;GO:0007416
NLGN1,,Actin_spine;Adhesion_PSD,False,"PSD Scaffolds, Trans-Synaptic Adhesion & Senescence/Survival",GO:0061001;hsa04514;GO:0007416;GO:0097104
NRXN1,,Adhesion_PSD,False,"PSD Scaffolds, Trans-Synaptic Adhesion & Senescence/Survival",hsa04514;GO:0007416;GO:0097104
CDH2,,Adhesion_PSD,False,"PSD Scaffolds, Trans-Synaptic Adhesion & Senescence/Survival",hsa04514;GO:0007416
BCL2,,TrkB_mTOR,False,"PSD Scaffolds, Trans-Synaptic Adhesion & Senescence/Survival",hsa04722;hsa04151
BCL2L1,,TrkB_mTOR,False,"PSD Scaffolds, Trans-Synaptic Adhesion & Senescence/Survival",hsa04151
BAD,,TrkB_mTOR,False,"PSD Scaffolds, Trans-Synaptic Adhesion & Senescence/Survival",hsa04722;hsa04151
TP53,,TrkB_mTOR,False,"PSD Scaffolds, Trans-Synaptic Adhesion & Senescence/Survival",hsa04722;hsa04151
CDKN1A,,TrkB_mTOR,False,"PSD Scaffolds, Trans-Synaptic Adhesion & Senescence/Survival",hsa04151
CYP2D6,,Not_in_backbone,True,Pharmacological Anchor (Absent from GMT),
SLC7A11,,Not_in_backbone,True,Pharmacological Anchor (Absent from GMT),
PDE4A,,Not_in_backbone,True,Pharmacological Anchor (Absent from GMT),
PDE4B,,Not_in_backbone,True,Pharmacological Anchor (Absent from GMT),
PDE4C,,Not_in_backbone,True,Pharmacological Anchor (Absent from GMT),
PDE4D,,Not_in_backbone,True,Pharmacological Anchor (Absent from GMT),
SLC6A4,,Not_in_backbone,True,Pharmacological Anchor (Absent from GMT),
SIGMAR1,,Not_in_backbone,True,Pharmacological Anchor (Absent from GMT),
SIRT1,,Not_in_backbone,True,Pharmacological Anchor (Absent from GMT),
PPARGC1A,,Not_in_backbone,True,Pharmacological Anchor (Absent from GMT),
NAMPT,,Not_in_backbone,True,Pharmacological Anchor (Absent from GMT),
PCYT1A,,Not_in_backbone,True,Pharmacological Anchor (Absent from GMT),
CHKA,,Not_in_backbone,True,Pharmacological Anchor (Absent from GMT),
SYP,,Not_in_backbone,True,Pharmacological Anchor (Absent from GMT),
SNAP25,,Not_in_backbone,True,Pharmacological Anchor (Absent from GMT),"""
Path("data/cgr_anchors.csv").write_text(ANCHORS_CSV + "\n", encoding="utf-8")
anchors_df = pd.read_csv("data/cgr_anchors.csv")
anchor_genes = sorted(anchors_df["symbol"].dropna().unique().tolist())
anchor_set = set(anchor_genes)
print(f"[OK] Loaded {len(anchor_genes)} embedded CGR + Pharm anchor genes.")

CGR_MOLECULES = {
    "Core_DXM":       "dextromethorphan",
    "Core_Piracetam": "piracetam",
    "Core_CYP2D6i":   CYP2D6_INHIBITOR,
    "Bacopa":         "bacoside A",
    "cAMP_Forskolin": "forskolin",
    "cAMP_Sceletium": "mesembrine",
    "Citicoline":     "citicoline",
    "Omega3_DHA":     "docosahexaenoic acid",
    "Fisetin":        "fisetin",
    "L_Glutamine":    "L-glutamine",
    "NAC":            "acetylcysteine",
    "NMN":            "nicotinamide mononucleotide",
}

# Optional additional query aliases for poorly-annotated naturals (tried in order).
CHEMBL_QUERY_ALIASES = {
    "Bacopa":         ["bacoside A", "bacopaside I", "bacoside A3"],
    "cAMP_Sceletium": ["mesembrine"],
    "cAMP_Forskolin": ["forskolin", "colforsin"],
}

# ---------------------------------------------------------------------------
# Curated MECHANISTIC targets (renamed from CURATED_METABOLIC_TARGETS).
# Now also encodes the established targets of the three ChEMBL zero-hit
# components (Forskolin/Sceletium/Bacopa) and the CYP2D6 PK node.
# All of these symbols already exist in the 121-gene anchor table.
# ---------------------------------------------------------------------------
CURATED_MECHANISTIC_TARGETS = {
    "Core_CYP2D6i":   ["CYP2D6"],                                   # PK gate (sub-5.5 pChEMBL, proximity cannot capture it)
    "cAMP_Forskolin": ["ADCY1", "ADCY8"],                           # direct adenylyl-cyclase activator
    "cAMP_Sceletium": ["PDE4A", "PDE4B", "PDE4C", "PDE4D", "SLC6A4"],# PDE4 inhibition + SERT
    "Bacopa":         ["BDNF", "NTRK2", "CREBBP", "CAMK2A"],         # neurotrophin/CREB axis
    "Core_Piracetam": ["GRIA1", "GRIA2", "GRIA3", "GRIA4"],
    "L_Glutamine":    ["GLUL", "GLS", "GLS2", "SLC1A1", "SLC1A2", "SLC1A3",
                       "SLC38A1", "SLC38A2", "SLC38A3", "SLC17A6", "SLC17A7", "SLC17A8"],
    "NAC":            ["SLC7A11", "GRIN1", "GRIN2A", "GRIN2B"],
    "NMN":            ["NAMPT", "SIRT1", "PPARGC1A"],
    "Citicoline":     ["PCYT1A", "CHKA"],
    "Omega3_DHA":     ["PLA2G4A", "PLA2G4B", "PLA2G4C", "PLA2G4D", "PLA2G4E",
                       "PLA2G4F", "DLG4", "GRIA1", "GRIA2"],
}
# Backward-compat alias (some downstream code / notebooks reference old name)
CURATED_METABOLIC_TARGETS = CURATED_MECHANISTIC_TARGETS

DISORDERS = {
    "MDD": "MONDO_0002009",
    "BIP": "MONDO_0004985",
    "SCZ": "MONDO_0005090",
    "OCD": "MONDO_0008114",
}
print(f"[OK] Config ready | pseudo-pChEMBL grid={PSEUDO_PCHEMBL_GRID} | PK penalty={PK_DXM_PENALTY} | N_PERM={N_PERM}")

[OK] Loaded 121 embedded CGR + Pharm anchor genes.
[OK] Config ready | pseudo-pChEMBL grid=[5.5, 6.0, 6.5] | PK penalty=1.0 | N_PERM=200


In [14]:
# @title 2. Query ChEMBL (cached) for Quantitative Human Bioactivity + Full Heteromer Subunit Resolution
SESSION = requests.Session()
SESSION.headers.update({"Accept": "application/json", "User-Agent": "CGR-Network-Pruning/2.0"})
CHEMBL = "https://www.ebi.ac.uk/chembl/api/data"

def get_chembl_id(name):
    def _exact():
        return SESSION.get(f"{CHEMBL}/molecule.json",
                           params={"pref_name__iexact": name, "limit": 5}, timeout=25).json()
    r = cached_json(f"mol_exact::{name}", _exact)
    mols = r.get("molecules", [])
    if not mols:
        def _search():
            return SESSION.get(f"{CHEMBL}/molecule/search.json",
                               params={"q": name, "limit": 5}, timeout=25).json()
        r = cached_json(f"mol_search::{name}", _search)
        mols = r.get("molecules", [])
    if not mols:
        return None
    mols_sorted = sorted(mols, key=lambda m: float(m.get("max_phase") or 0), reverse=True)
    m0 = mols_sorted[0]
    parent = (m0.get("molecule_hierarchy") or {}).get("parent_chembl_id")
    return parent or m0.get("molecule_chembl_id")

def resolve_chembl_id_with_aliases(label, primary_name):
    """Try the primary query name, then any curated aliases."""
    names = [primary_name] + [a for a in CHEMBL_QUERY_ALIASES.get(label, []) if a != primary_name]
    for nm in names:
        cid = get_chembl_id(nm)
        if cid:
            return cid, nm
    return None, primary_name

def fetch_chembl_targets(chembl_id, pchembl_min=PCHEMBL_CUTOFF):
    rows, offset = [], 0
    while True:
        def _page(off=offset):
            return SESSION.get(f"{CHEMBL}/activity.json", params={
                "molecule_chembl_id": chembl_id,
                "pchembl_value__gte": pchembl_min,
                "target_organism": "Homo sapiens",
                "limit": 500, "offset": off,
            }, timeout=30).json()
        r = cached_json(f"act_mol::{chembl_id}::{pchembl_min}::{offset}", _page)
        acts = r.get("activities", [])
        for act in acts:
            if act.get("standard_type") in {"Ki", "IC50", "EC50", "Kd"} and act.get("pchembl_value"):
                rows.append({
                    "target_chembl_id": act.get("target_chembl_id"),
                    "target_name":      act.get("target_pref_name"),
                    "pchembl":          float(act["pchembl_value"]),
                })
        if not r.get("page_meta", {}).get("next") or len(acts) < 500 or offset >= 1500:
            break
        offset += 500
    if not rows:
        return pd.DataFrame()
    return pd.DataFrame(rows).groupby(["target_chembl_id", "target_name"], as_index=False)["pchembl"].median()

def resolve_target_symbols(target_chembl_ids):
    """
    FIX: map each target_chembl_id to the FULL LIST of constituent gene symbols,
    so heteromeric complexes (e.g. CHEMBL1907604 = GRIN1/GRIN2A) keep every subunit.
    """
    tid_to_syms = {}
    sym_to_single_tid = {}
    for tid in sorted(set(target_chembl_ids)):
        def _fetch(tid=tid):
            return SESSION.get(f"{CHEMBL}/target/{tid}.json", timeout=20).json()
        try:
            r = cached_json(f"target::{tid}", _fetch)
        except Exception:
            continue
        t_type = r.get("target_type", "")
        syms = []
        for comp in r.get("target_components", []):
            for syn in comp.get("target_component_synonyms", []):
                if syn.get("syn_type") == "GENE_SYMBOL":
                    syms.append(syn["component_synonym"])
                    break  # one GENE_SYMBOL per component, but keep ALL components
        syms = sorted(set(syms))
        if syms:
            tid_to_syms[tid] = syms
            if t_type == "SINGLE PROTEIN" and len(syms) == 1:
                sym_to_single_tid.setdefault(syms[0], tid)
        time.sleep(0.01)
    return tid_to_syms, sym_to_single_tid

print("Querying ChEMBL for CGR components (cached)...")
drug_targets_binding = {}
all_tids = set()
lookup_summary = []

for label, mol_name in CGR_MOLECULES.items():
    cid, used_name = resolve_chembl_id_with_aliases(label, mol_name)
    tdf = fetch_chembl_targets(cid) if cid else pd.DataFrame()
    lookup_summary.append({
        "component": label, "query": used_name, "chembl_id": cid, "n_raw_targets": len(tdf),
    })
    if not tdf.empty:
        drug_targets_binding[label] = tdf
        all_tids.update(tdf["target_chembl_id"].tolist())

tid_to_syms, sym_to_single_tid = resolve_target_symbols(all_tids)

binding_export = []
for label, tdf in list(drug_targets_binding.items()):
    tdf = tdf.copy()
    tdf["symbol"] = tdf["target_chembl_id"].map(tid_to_syms)          # list per tid
    tdf = tdf.explode("symbol").dropna(subset=["symbol"])            # expand heteromers
    tdf = tdf.groupby("symbol", as_index=False).agg(
        pchembl=("pchembl", "max"),
        target_chembl_id=("target_chembl_id", "first"),
        target_name=("target_name", "first"),
    )
    if tdf.empty:
        del drug_targets_binding[label]
    else:
        drug_targets_binding[label] = tdf
        tmp = tdf.copy(); tmp["component"] = label
        binding_export.append(tmp)

# Explicitly log zero-ChEMBL-hit components (retained downstream, never silently dropped)
zero_hit_components = [lbl for lbl in CGR_MOLECULES if lbl not in drug_targets_binding]
print(f"[INFO] Zero binding-hit components (pChEMBL >= {PCHEMBL_CUTOFF}): "
      f"{zero_hit_components if zero_hit_components else 'none'}")
print("[INFO] These are retained via CURATED_MECHANISTIC_TARGETS in the metabolic/PK modes "
      "and kept in all tables with n_targets=0 in binding-only mode.")

binding_profiles_df = pd.concat(binding_export, ignore_index=True) if binding_export else pd.DataFrame()
binding_profiles_df.to_csv("results/results_chembl_binding_profiles.csv", index=False)
lookup_summary_df = pd.DataFrame(lookup_summary)
lookup_summary_df.to_csv("results/results_chembl_lookup_summary.csv", index=False)
display(lookup_summary_df)
print(f"[OK] Resolved {len(binding_profiles_df)} component-target pairs across "
      f"{len(drug_targets_binding)} binding-active components "
      f"(heteromer subunits fully expanded).")

Querying ChEMBL for CGR components (cached)...
[INFO] Zero binding-hit components (pChEMBL >= 5.5): ['Core_Piracetam', 'Bacopa', 'cAMP_Forskolin', 'cAMP_Sceletium', 'Citicoline', 'L_Glutamine', 'NAC']
[INFO] These are retained via CURATED_MECHANISTIC_TARGETS in the metabolic/PK modes and kept in all tables with n_targets=0 in binding-only mode.


,component,query,chembl_id,n_raw_targets
0,Core_DXM,dextromethorphan,CHEMBL52440,5
1,Core_Piracetam,piracetam,CHEMBL36715,0
2,Core_CYP2D6i,bupropion,CHEMBL894,6
3,Bacopa,bacoside A,CHEMBL4297945,0
4,cAMP_Forskolin,forskolin,CHEMBL4782169,0
5,cAMP_Sceletium,mesembrine,None,0
6,Citicoline,citicoline,CHEMBL1231700,0
7,Omega3_DHA,docosahexaenoic acid,CHEMBL367149,2
8,Fisetin,fisetin,CHEMBL31574,15
9,L_Glutamine,L-glutamine,CHEMBL930,0


[OK] Resolved 39 component-target pairs across 5 binding-active components (heteromer subunits fully expanded).


In [15]:
# @title 3. Query Open Targets Disorder Weights & Build STRING Human PPI Network (both cached)
def fetch_opentargets_weights(disease_id, gene_symbols):
    url = "https://api.platform.opentargets.org/api/v4/graphql"
    query = """
    query($efoId: String!, $size: Int!) {
      disease(efoId: $efoId) {
        associatedTargets(page: {index: 0, size: $size}) {
          rows { target { approvedSymbol } score }
        }
      }
    }
    """
    def _fetch():
        return SESSION.post(url, json={"query": query,
                            "variables": {"efoId": disease_id, "size": 3000}}, timeout=45).json()
    r = cached_json(f"ot::{disease_id}", _fetch)
    rows = ((r.get("data") or {}).get("disease") or {}).get("associatedTargets", {}).get("rows", [])
    scores = {row["target"]["approvedSymbol"]: float(row["score"]) for row in rows}
    return {g: scores.get(g, 0.0) for g in gene_symbols}

print("Fetching Open Targets disorder weights (cached)...")
disorder_weights = {}
for d, eid in DISORDERS.items():
    w = fetch_opentargets_weights(eid, anchor_genes)
    disorder_weights[d] = w
    nonzero = sum(1 for v in w.values() if v > 0)
    print(f"  {d} ({eid}): {nonzero}/{len(anchor_genes)} anchors with score > 0")

# Union of all nodes (anchors + binding targets + curated mechanistic targets)
all_nodes = set(anchor_genes)
for tdf in drug_targets_binding.values():
    all_nodes.update(tdf["symbol"].tolist())
for syms in CURATED_MECHANISTIC_TARGETS.values():
    all_nodes.update(syms)
all_nodes = sorted(all_nodes)

print(f"\nQuerying STRING PPI network for {len(all_nodes)} human proteins "
      f"(score >= {STRING_SCORE_CUTOFF}, cached)...")

def fetch_string_edges(nodes, score_cutoff):
    key = "string::" + hashlib.md5(("|".join(nodes) + f"::{score_cutoff}").encode()).hexdigest()
    def _fetch():
        resp = SESSION.post("https://string-db.org/api/json/network", data={
            "identifiers": "\r".join(nodes),
            "species": 9606,
            "required_score": score_cutoff,
            "caller_identity": "cgr_network_pruning_colab",
        }, timeout=90)
        resp.raise_for_status()
        return resp.json()
    return cached_json(key, _fetch)

edges = fetch_string_edges(all_nodes, STRING_SCORE_CUTOFF)

G = nx.Graph()
for edge in edges:
    G.add_edge(edge["preferredName_A"], edge["preferredName_B"], weight=float(edge["score"]))
print(f"[OK] STRING graph built: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges.")

Fetching Open Targets disorder weights (cached)...
  MDD (MONDO_0002009): 73/121 anchors with score > 0
  BIP (MONDO_0004985): 71/121 anchors with score > 0
  SCZ (MONDO_0005090): 65/121 anchors with score > 0
  OCD (MONDO_0008114): 53/121 anchors with score > 0

Querying STRING PPI network for 150 human proteins (score >= 700, cached)...
[OK] STRING graph built: 145 nodes, 904 edges.


In [16]:
# @title 4. Affinity-Weighted LOO Ablation — 3 modes + pseudo-pChEMBL sweep + degree-matched null
anchor_in_G = [g for g in anchor_genes if g in G]
dist_cache = {a: nx.single_source_shortest_path_length(G, a) for a in anchor_in_G}

# Midpoint for Hill/Langmuir occupancy scaling so affinity & PK attenuation affect direct (d=0) hits.
PCHEMBL_OCC_MIDPOINT = 4.5 # Hardcoded

def weighted_module_coverage(target_dict, disorder_score_map):
    active_targets = {g: w for g, w in target_dict.items() if g in G}
    if not active_targets:
        return 0.0
    num, den = 0.0, 0.0
    for anchor, d_score in disorder_score_map.items():
        if anchor not in dist_cache or d_score <= 0:
            continue
        dmap = dist_cache[anchor]
        best_score = -1.0
        for t, pchembl in active_targets.items():
            d = dmap.get(t)
            if d is not None:
                eff_d = d / max(1.0, (pchembl - 4.5))
                occ = 1.0 / (1.0 + 10.0 ** (PCHEMBL_OCC_MIDPOINT - pchembl))
                score = occ / (1.0 + eff_d)
                if score > best_score:
                    best_score = score
        if best_score >= 0.0:
            num += d_score * best_score
            den += d_score
    return num / den if den > 0 else 0.0

def build_component_map(include_mechanistic=False, pseudo_pchembl=METABOLIC_PSEUDO_PCHEMBL):
    """Always seeds ALL CGR_MOLECULES labels so zero-hit components are retained."""
    comp_map = {label: {} for label in CGR_MOLECULES}
    for label, tdf in drug_targets_binding.items():
        comp_map[label] = dict(zip(tdf["symbol"], tdf["pchembl"]))
    if include_mechanistic:
        for label, syms in CURATED_MECHANISTIC_TARGETS.items():
            cur = comp_map.get(label, {}).copy()
            for s in syms:
                cur[s] = max(cur.get(s, 0.0), pseudo_pchembl)
            comp_map[label] = cur
    return comp_map

def _aggregate(comp_map, exclude=None, attenuate=None, penalty=0.0):
    agg = {}
    for label, d in comp_map.items():
        if label == exclude:
            continue
        for s, p in d.items():
            eff = max(0.0, p - penalty) if (attenuate and label in attenuate) else p
            agg[s] = max(agg.get(s, 0.0), eff)
    return agg

def run_loo(mode="binding_only", pseudo_pchembl=METABOLIC_PSEUDO_PCHEMBL):
    """
    mode in {binding_only, with_metabolic, with_metabolic_and_pk}
    PK mode: dropping Core_CYP2D6i attenuates Core_DXM target pChEMBL by PK_DXM_PENALTY,
             modelling loss of CYP2D6 inhibition (faster DXM -> dextrorphan clearance).
    """
    include_mech = mode in ("with_metabolic", "with_metabolic_and_pk")
    pk = (mode == "with_metabolic_and_pk")
    comp_map = build_component_map(include_mechanistic=include_mech, pseudo_pchembl=pseudo_pchembl)
    full_stack = _aggregate(comp_map)

    rows = []
    for disorder, d_map in disorder_weights.items():
        base_cov = weighted_module_coverage(full_stack, d_map)
        for drop_label in sorted(comp_map.keys()):
            attenuate = {"Core_DXM"} if (pk and drop_label == "Core_CYP2D6i") else None
            reduced = _aggregate(comp_map, exclude=drop_label,
                                 attenuate=attenuate, penalty=PK_DXM_PENALTY)
            loo_cov = weighted_module_coverage(reduced, d_map)
            rows.append({
                "mode": mode, "disorder": disorder, "dropped": drop_label,
                "n_targets": len(comp_map[drop_label]),
                "full_coverage": round(base_cov, 5),
                "loo_coverage": round(loo_cov, 5),
                "unique_delta": round(base_cov - loo_cov, 5),
            })
    return pd.DataFrame(rows), full_stack

# ---- Run all three modes ---------------------------------------------------
loo_binding_df, full_stack_binding = run_loo("binding_only")
loo_full_df,    full_stack_all     = run_loo("with_metabolic")
loo_pk_df,      full_stack_pk      = run_loo("with_metabolic_and_pk")

loo_binding_df.to_csv("results/results_loo_ablation_binding_only.csv", index=False)
loo_full_df.to_csv("results/results_loo_ablation_with_metabolic.csv", index=False)
loo_pk_df.to_csv("results/results_loo_ablation_with_metabolic_and_pk.csv", index=False)

summary_binding = loo_binding_df.groupby("dropped", as_index=False)["unique_delta"].mean().rename(columns={"unique_delta": "mean_delta_binding_only"})
summary_full    = loo_full_df.groupby("dropped", as_index=False)["unique_delta"].mean().rename(columns={"unique_delta": "mean_delta_with_metabolic"})
summary_pk      = loo_pk_df.groupby("dropped", as_index=False)["unique_delta"].mean().rename(columns={"unique_delta": "mean_delta_with_pk"})
cgr_lite_summary = summary_full.merge(summary_pk, on="dropped", how="outer").merge(summary_binding, on="dropped", how="outer")
cgr_lite_summary = cgr_lite_summary.sort_values("mean_delta_with_pk", ascending=False)
cgr_lite_summary.to_csv("results/results_cgr_lite_summary.csv", index=False)
print("=== CGR-Lite Component Ranking (higher mean unique_delta = more irreplaceable) ===")
display(cgr_lite_summary)

# ---- Sensitivity sweep over the curated pseudo-pChEMBL assumption ----------
print(f"\nPseudo-pChEMBL rank-stability sweep over {PSEUDO_PCHEMBL_GRID} ...")
sens_rows = []
for pv in PSEUDO_PCHEMBL_GRID:
    df_pv, _ = run_loo("with_metabolic", pseudo_pchembl=pv)
    s = df_pv.groupby("dropped")["unique_delta"].mean()
    for comp, val in s.items():
        sens_rows.append({"pseudo_pchembl": pv, "dropped": comp, "mean_delta": round(float(val), 5)})
sens_long = pd.DataFrame(sens_rows)
sens_pivot = sens_long.pivot(index="dropped", columns="pseudo_pchembl", values="mean_delta")
for pv in PSEUDO_PCHEMBL_GRID:
    sens_pivot[f"rank@{pv}"] = sens_pivot[pv].rank(ascending=False, method="min").astype(int)
rank_cols = [f"rank@{pv}" for pv in PSEUDO_PCHEMBL_GRID]
sens_pivot["rank_range"] = sens_pivot[rank_cols].max(axis=1) - sens_pivot[rank_cols].min(axis=1)
sens_pivot = sens_pivot.sort_values(f"rank@{METABOLIC_PSEUDO_PCHEMBL}")
sens_pivot.to_csv("results/results_pseudo_pchembl_sensitivity.csv")
print("[OK] Rank stability (rank_range = 0 => rank invariant to pseudo-affinity assumption):")
display(sens_pivot)

# ---- Degree-matched permutation null (empirical z-score & p-value) ---------
def build_degree_bins(G, n_bins=10):
    nodes = list(G.nodes())
    degs = np.array([G.degree(n) for n in nodes])
    order = np.argsort(degs)
    ranks = np.empty(len(nodes), dtype=int); ranks[order] = np.arange(len(nodes))
    node_bin, bins = {}, {}
    for i, n in enumerate(nodes):
        b = int(ranks[i] * n_bins / max(1, len(nodes)))
        node_bin[n] = b
        bins.setdefault(b, []).append(n)
    return node_bin, bins

def permutation_test(mode="with_metabolic", pseudo_pchembl=METABOLIC_PSEUDO_PCHEMBL, n_perm=N_PERM):
    rng = np.random.default_rng(RANDOM_SEED)
    node_bin, bins = build_degree_bins(G)
    all_graph_nodes = list(G.nodes())
    include_mech = mode in ("with_metabolic", "with_metabolic_and_pk")
    pk = (mode == "with_metabolic_and_pk")
    comp_map = build_component_map(include_mechanistic=include_mech, pseudo_pchembl=pseudo_pchembl)
    full_stack_obs = _aggregate(comp_map)
    full_cov_obs = {dname: weighted_module_coverage(full_stack_obs, dm)
                    for dname, dm in disorder_weights.items()}

    results = []
    for comp_label, cdict in comp_map.items():
        tg = {s: p for s, p in cdict.items() if s in G}
        k = len(tg)

        is_pk_gate = pk and (comp_label == "Core_CYP2D6i")
        rest_unattenuated = _aggregate(comp_map, exclude=comp_label)
        rest_baseline = (
            _aggregate(comp_map, exclude=comp_label, attenuate={"Core_DXM"}, penalty=PK_DXM_PENALTY)
            if is_pk_gate else rest_unattenuated
        )
        rest_cov = {dname: weighted_module_coverage(rest_baseline, dm)
                    for dname, dm in disorder_weights.items()}

        # Observed marginal delta matches run_loo(mode) exactly
        obs_deltas = [full_cov_obs[dname] - rest_cov[dname] for dname in disorder_weights]
        obs = float(np.mean(obs_deltas)) if obs_deltas else 0.0

        if k == 0:
            results.append({"mode": mode, "component": comp_label, "n_targets": 0,
                            "observed_delta": round(obs, 5), "null_mean": np.nan,
                            "null_std": np.nan, "z_score": np.nan, "p_value": np.nan})
            continue

        pvals = list(tg.values())
        null = np.empty(n_perm)
        for j in range(n_perm):
            rand_targets = {}
            used = set()
            for i, s in enumerate(tg):
                pool = bins.get(node_bin.get(s), all_graph_nodes)
                avail = [n for n in pool if n not in used]
                if not avail:
                    avail = [n for n in all_graph_nodes if n not in used] or pool
                rs = rng.choice(avail)
                used.add(rs)
                rand_targets[rs] = max(rand_targets.get(rs, 0.0), pvals[i])

            # In PK mode for Core_CYP2D6i, a random target set only rescues DXM if it hits CYP2D6
            base_for_draw = (
                rest_unattenuated if (is_pk_gate and "CYP2D6" in rand_targets) else rest_baseline
            )
            nd = []
            for dname, dm in disorder_weights.items():
                full = dict(base_for_draw)
                for s, p in rand_targets.items():
                    full[s] = max(full.get(s, 0.0), p)
                nd.append(weighted_module_coverage(full, dm) - rest_cov[dname])
            null[j] = np.mean(nd)

        mu, sd = float(null.mean()), float(null.std())
        z = (obs - mu) / sd if sd > 0 else 0.0
        p = float((np.sum(null >= obs) + 1) / (n_perm + 1))
        results.append({"mode": mode, "component": comp_label, "n_targets": k,
                        "observed_delta": round(obs, 5), "null_mean": round(mu, 5),
                        "null_std": round(sd, 5), "z_score": round(z, 3), "p_value": round(p, 4)})
    return pd.DataFrame(results).sort_values("z_score", ascending=False)

print(f"\nDegree-matched permutation null model (N_PERM={N_PERM} per component)...")
perm_full_df = permutation_test("with_metabolic")
perm_pk_df   = permutation_test("with_metabolic_and_pk")
perm_full_df.to_csv("results/results_permutation_null_with_metabolic.csv", index=False)
perm_pk_df.to_csv("results/results_permutation_null_with_metabolic_and_pk.csv", index=False)
print("[OK] Permutation z-scores / empirical p-values (with_metabolic):")
display(perm_full_df)
print("[OK] Permutation z-scores / empirical p-values (with_metabolic_and_pk):")
display(perm_pk_df)

=== CGR-Lite Component Ranking (higher mean unique_delta = more irreplaceable) ===


,dropped,mean_delta_with_metabolic,mean_delta_with_pk,mean_delta_binding_only
3,Core_DXM,0.043298,0.043298,0.160685
2,Core_CYP2D6i,0.000015,0.033095,0.000045
4,Core_Piracetam,0.028125,0.028125,0.000000
11,cAMP_Sceletium,0.018403,0.018403,0.000000
6,L_Glutamine,0.014038,0.014038,0.000000
0,Bacopa,0.007485,0.007485,0.000000
8,NMN,0.005995,0.005995,0.000050
7,NAC,0.004920,0.004920,0.000000
5,Fisetin,0.004593,0.004593,0.005857
9,Omega3_DHA,0.003785,0.003785,0.000000



Pseudo-pChEMBL rank-stability sweep over [5.5, 6.0, 6.5] ...
[OK] Rank stability (rank_range = 0 => rank invariant to pseudo-affinity assumption):


pseudo_pchembl,5.5,6.0,6.5,rank@5.5,rank@6.0,rank@6.5,rank_range
dropped,,,,,,,
Core_DXM,0.07386,0.04330,0.03238,1,1,1,0
Core_Piracetam,0.02373,0.02813,0.02406,2,2,2,0
cAMP_Sceletium,0.01553,0.01840,0.01574,3,3,3,0
L_Glutamine,0.01185,0.01404,0.01332,4,4,4,0
Bacopa,0.00556,0.00749,0.00821,5,5,6,1
NMN,0.00487,0.00600,0.00672,6,6,8,2
NAC,0.00322,0.00492,0.00877,8,7,5,3
Fisetin,0.00459,0.00459,0.00304,7,8,9,2
Omega3_DHA,0.00313,0.00379,0.00685,9,9,7,2



Degree-matched permutation null model (N_PERM=200 per component)...
[OK] Permutation z-scores / empirical p-values (with_metabolic):


,mode,component,n_targets,observed_delta,null_mean,null_std,z_score,p_value
1,with_metabolic,Core_Piracetam,4,0.02812,0.00516,0.00606,3.788,0.0199
5,with_metabolic,cAMP_Sceletium,4,0.01840,0.00488,0.00438,3.088,0.0050
0,with_metabolic,Core_DXM,9,0.04330,0.01519,0.01356,2.073,0.0746
10,with_metabolic,NAC,4,0.00492,0.00232,0.00184,1.412,0.1095
3,with_metabolic,Bacopa,4,0.00748,0.00327,0.00299,1.405,0.1144
11,with_metabolic,NMN,4,0.00599,0.00301,0.00314,0.948,0.2139
9,with_metabolic,L_Glutamine,11,0.01404,0.00927,0.00895,0.533,0.1940
6,with_metabolic,Citicoline,2,0.00001,0.00006,0.00018,-0.269,0.2438
7,with_metabolic,Omega3_DHA,9,0.00379,0.00529,0.00356,-0.423,0.5721
4,with_metabolic,cAMP_Forskolin,2,0.00146,0.00494,0.00791,-0.440,0.4030


[OK] Permutation z-scores / empirical p-values (with_metabolic_and_pk):


,mode,component,n_targets,observed_delta,null_mean,null_std,z_score,p_value
1,with_metabolic_and_pk,Core_Piracetam,4,0.02812,0.00516,0.00606,3.788,0.0199
5,with_metabolic_and_pk,cAMP_Sceletium,4,0.01840,0.00488,0.00438,3.088,0.0050
0,with_metabolic_and_pk,Core_DXM,9,0.04330,0.01519,0.01356,2.073,0.0746
2,with_metabolic_and_pk,Core_CYP2D6i,7,0.03309,0.01030,0.01489,1.531,0.2040
10,with_metabolic_and_pk,NAC,4,0.00492,0.00232,0.00184,1.412,0.1095
3,with_metabolic_and_pk,Bacopa,4,0.00748,0.00327,0.00299,1.405,0.1144
11,with_metabolic_and_pk,NMN,4,0.00599,0.00301,0.00314,0.948,0.2139
9,with_metabolic_and_pk,L_Glutamine,11,0.01404,0.00927,0.00895,0.533,0.1940
6,with_metabolic_and_pk,Citicoline,2,0.00001,0.00006,0.00018,-0.269,0.2438
7,with_metabolic_and_pk,Omega3_DHA,9,0.00379,0.00529,0.00356,-0.423,0.5721


In [17]:
# @title 5. Screen ChEMBL for Single FDA-Approved (Phase 4) Multi-Target Replacements
# Fixes: (1) type-safe max_phase>=4.0, (2) mechanism.json + sorted activity.json two-tier,
#        (3) screen targets drawn from CGR-target ∩ 121-anchor intersection, ranked by mean
#            Open Targets disorder weight (pChEMBL as tie-breaker) to avoid arbitrary tie slicing.

mean_ot_weight = {
    s: float(np.mean([disorder_weights[d].get(s, 0.0) for d in DISORDERS]))
    for s in anchor_set
}
screen_pool = {s: p for s, p in full_stack_all.items() if s in anchor_set and s != "CYP2D6"}
top_targets = sorted(
    screen_pool.items(),
    key=lambda x: (mean_ot_weight.get(x[0], 0.0), x[1]),
    reverse=True,
)[:12]
print("Anchor-intersecting screen targets (ranked by mean Open Targets weight):")
for sym, pv in top_targets:
    print(f"  {sym:<10} mean_OT={mean_ot_weight.get(sym, 0.0):.4f}  pChEMBL={pv:.2f}")

def fetch_fda_drugs_for_target(tid):
    """Two-tier: curated mechanism drugs first, then high-potency sorted activities."""
    hits = {}  # mcid -> {"via":..., "pchembl":float|None}
    # Tier 1 — curated clinical mechanisms (approved/validated drug–target links)
    def _mech():
        return SESSION.get(f"{CHEMBL}/mechanism.json",
                           params={"target_chembl_id": tid, "limit": 500}, timeout=30).json()
    mech = cached_json(f"mech::{tid}", _mech)
    for mh in mech.get("mechanisms", []):
        mcid = mh.get("molecule_chembl_id")
        if mcid:
            hits.setdefault(mcid, {"via": "mechanism", "pchembl": None})
    # Tier 2 — high-potency, descending-sorted bioactivities (no preclinical dilution)
    def _act():
        return SESSION.get(f"{CHEMBL}/activity.json", params={
            "target_chembl_id": tid, "pchembl_value__gte": FDA_ACT_PCHEMBL_MIN,
            "order_by": "-pchembl_value", "limit": 300,
        }, timeout=30).json()
    act = cached_json(f"act_tgt::{tid}::{FDA_ACT_PCHEMBL_MIN}", _act)
    for a in act.get("activities", []):
        mcid, pv = a.get("molecule_chembl_id"), a.get("pchembl_value")
        if mcid and pv:
            cur = hits.get(mcid, {"via": "activity", "pchembl": 0.0})
            cur["pchembl"] = max(cur.get("pchembl") or 0.0, float(pv))
            hits[mcid] = cur
    return hits

fda_hits = {}       # mcid -> set(symbols)
fda_pchembl = {}    # mcid -> {sym: pchembl}
for sym, _ in top_targets:
    tid = sym_to_single_tid.get(sym)
    if not tid:
        def _tsearch(sym=sym):
            return SESSION.get(f"{CHEMBL}/target/search.json",
                               params={"q": sym, "limit": 5}, timeout=20).json()
        t_res = cached_json(f"tgt_search::{sym}", _tsearch)
        human = [t["target_chembl_id"] for t in t_res.get("targets", [])
                 if t.get("organism") == "Homo sapiens" and t.get("target_type") == "SINGLE PROTEIN"]
        tid = human[0] if human else None
    if not tid:
        continue
    for mcid, info in fetch_fda_drugs_for_target(tid).items():
        fda_hits.setdefault(mcid, set()).add(sym)
        # mechanism-only hits (no pChEMBL) get the activity floor as a conservative proxy
        pv = info["pchembl"] if info["pchembl"] is not None else FDA_ACT_PCHEMBL_MIN
        fda_pchembl.setdefault(mcid, {})[sym] = max(fda_pchembl.get(mcid, {}).get(sym, 0.0), pv)

multi_mcids = [m for m, syms in fda_hits.items() if len(syms) >= 2]
print(f"\nChecking max_phase >= 4.0 for {len(multi_mcids)} multi-target ChEMBL molecules...")

def get_molecule(mcid):
    def _fetch():
        return SESSION.get(f"{CHEMBL}/molecule/{mcid}.json", timeout=20).json()
    return cached_json(f"molrec::{mcid}", _fetch)

candidates = []
for mcid in multi_mcids:
    try:
        m = get_molecule(mcid)
    except Exception:
        continue
    # (1) type-safe phase filter: ChEMBL may return "4.0" (str), 4.0 (float) or 4 (int)
    if float(m.get("max_phase") or 0) >= 4.0:
        tdict = fda_pchembl[mcid]
        covs = {d: round(weighted_module_coverage(tdict, d_map), 4)
                for d, d_map in disorder_weights.items()}
        candidates.append({
            "chembl_id": mcid,
            "drug_name": m.get("pref_name"),
            "max_phase": float(m.get("max_phase") or 0),
            "n_targets": len(fda_hits[mcid]),
            "targets_hit": ", ".join(sorted(fda_hits[mcid])),
            "mean_cov_4dis": round(float(np.mean(list(covs.values()))), 4),
            "cov_MDD": covs["MDD"], "cov_BIP": covs["BIP"],
            "cov_SCZ": covs["SCZ"], "cov_OCD": covs["OCD"],
        })

fda_df = pd.DataFrame(candidates)
if not fda_df.empty:
    fda_df = fda_df.sort_values(["n_targets", "mean_cov_4dis"], ascending=[False, False])
fda_df.to_csv("results/results_fda_replacements.csv", index=False)
print(f"[OK] {len(fda_df)} Phase-4 multi-target replacement candidates.")
display(fda_df.head(20))

# Bundle every CSV produced so far (summary .txt is appended in Cell 6).
zip_path = Path("cgr_network_pruning_results.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(Path("results").glob("*.csv")):
        zf.write(p, arcname=p.name)
print(f"[OK] Bundled CSV outputs into {zip_path} (summary added in next cell).")

Anchor-intersecting screen targets (ranked by mean Open Targets weight):
  GRIN2A     mean_OT=0.5866  pChEMBL=6.00
  SLC6A4     mean_OT=0.5838  pChEMBL=8.71
  GRIN2B     mean_OT=0.4512  pChEMBL=6.00
  GRIN1      mean_OT=0.4438  pChEMBL=6.00
  GRIN2D     mean_OT=0.4398  pChEMBL=5.78
  GRIN2C     mean_OT=0.4388  pChEMBL=5.78
  GRIA1      mean_OT=0.3514  pChEMBL=6.00
  GRIA3      mean_OT=0.2590  pChEMBL=6.00
  PDE4B      mean_OT=0.2565  pChEMBL=6.00
  GRIA4      mean_OT=0.2546  pChEMBL=6.00
  GRIA2      mean_OT=0.2544  pChEMBL=6.00
  SIGMAR1    mean_OT=0.2529  pChEMBL=6.46

Checking max_phase >= 4.0 for 33 multi-target ChEMBL molecules...
[OK] 2 Phase-4 multi-target replacement candidates.


,chembl_id,drug_name,max_phase,n_targets,targets_hit,mean_cov_4dis,cov_MDD,cov_BIP,cov_SCZ,cov_OCD
0,CHEMBL1201461,DEXTROMETHORPHAN POLISTIREX,4.0,2,"GRIN2A, SIGMAR1",0.6019,0.5791,0.6218,0.5707,0.6362
1,CHEMBL1256818,DEXTROMETHORPHAN HYDROBROMIDE,4.0,2,"GRIN2A, SIGMAR1",0.6019,0.5791,0.6218,0.5707,0.6362


[OK] Bundled CSV outputs into cgr_network_pruning_results.zip (summary added in next cell).


## Summary

In [18]:
# @title 6. Generate Detailed Human-Readable Summary (.txt), Append to Zip & Download
from datetime import datetime, timezone

def fmt_df(df, floatfmt="{:.5f}", max_rows=200):
    if df is None or len(df) == 0:
        return "    (empty)"
    d = df.copy().head(max_rows)
    for c in d.columns:
        if pd.api.types.is_float_dtype(d[c]):
            d[c] = d[c].map(lambda v: "" if pd.isna(v) else floatfmt.format(v))
    return d.to_string(index=False).replace("\n", "\n    ").rjust(0).__str__().replace("\n", "\n    ").rstrip() \
        .join(["    ", ""])  # indent block

def block(df, floatfmt="{:.5f}", max_rows=200):
    if df is None or len(df) == 0:
        return "    (empty)"
    d = df.copy().head(max_rows)
    for c in d.columns:
        if pd.api.types.is_float_dtype(d[c]):
            d[c] = d[c].map(lambda v: "" if pd.isna(v) else floatfmt.format(v))
    return "    " + d.to_string(index=False).replace("\n", "\n    ")

L = []
W = L.append
ts = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")

W("=" * 100)
W("CGR NETWORK PRUNING · LOO ABLATION · FDA REPLACEMENT SCREEN — DETAILED RUN SUMMARY")
W("=" * 100)
W(f"Generated         : {ts}")
W(f"Pipeline version  : CGR-Network-Pruning/2.0 (publication-grade, cached, 3-mode)")
W("")
W("-" * 100)
W("0. METHODS OVERVIEW")
W("-" * 100)
W("This pipeline queries ChEMBL (bioactivity + curated mechanisms), Open Targets (genetic")
W("association weights) and STRING (human PPI, score cutoff below) live, then runs an")
W("affinity-weighted leave-one-out (LOO) ablation of the CGR stack across MDD, BIP, SCZ and")
W("OCD. Component importance = mean 'unique_delta' = drop in disorder-weighted module")
W("coverage when a component is removed. Coverage uses network proximity from each disorder")
W("anchor to the nearest drug target, scaled by target affinity:")
W("")
W("    eff_distance = shortest_path(anchor, target) / max(1, pChEMBL - 4.5)")
W("    coverage     = Σ_anchor [ disorder_score · 1/(1+min_eff_distance) ] / Σ_anchor disorder_score")
W("")
W("Three sensitivity modes are reported:")
W("  • binding_only            : ChEMBL pChEMBL >= cutoff targets only.")
W("  • with_metabolic          : adds curated mechanistic/metabolic/structural targets.")
W("  • with_metabolic_and_pk   : as above, PLUS a CYP2D6 PK gate — dropping the CYP2D6")
W("                              inhibitor attenuates DXM target potency by PK_DXM_PENALTY,")
W("                              quantifying the inhibitor's true functional indispensability.")
W("")
W("-" * 100)
W("1. CONFIGURATION")
W("-" * 100)
W(f"  PCHEMBL_CUTOFF            = {PCHEMBL_CUTOFF}")
W(f"  STRING_SCORE_CUTOFF      = {STRING_SCORE_CUTOFF}")
W(f"  CYP2D6_INHIBITOR         = {CYP2D6_INHIBITOR}")
W(f"  METABOLIC_PSEUDO_PCHEMBL = {METABOLIC_PSEUDO_PCHEMBL}")
W(f"  PSEUDO_PCHEMBL_GRID      = {PSEUDO_PCHEMBL_GRID}")
W(f"  PK_DXM_PENALTY           = {PK_DXM_PENALTY}")
W(f"  N_PERM                   = {N_PERM}")
W(f"  FDA_ACT_PCHEMBL_MIN      = {FDA_ACT_PCHEMBL_MIN}")
W(f"  RANDOM_SEED              = {RANDOM_SEED}")
W(f"  Anchor genes (121 set)   = {len(anchor_genes)}")
W(f"  Disorders                = {', '.join(f'{k} ({v})' for k,v in DISORDERS.items())}")
W("")
W("-" * 100)
W("2. ChEMBL COMPOUND LOOKUP")
W("-" * 100)
W(f"  Binding-active components : {len(drug_targets_binding)} / {len(CGR_MOLECULES)}")
W(f"  Zero binding-hit (pChEMBL >= {PCHEMBL_CUTOFF}) components (RETAINED, not dropped):")
W(f"    {zero_hit_components if zero_hit_components else 'none'}")
W("  -> Zero-hit naturals (Forskolin/Sceletium/Bacopa) re-enter via CURATED_MECHANISTIC_TARGETS")
W("     and remain in all LOO tables (n_targets=0, unique_delta=0 in binding_only mode).")
W("  Heteromeric complexes: ALL constituent gene symbols are retained (explode before groupby),")
W("  so e.g. GRIN1/GRIN2A NMDA complexes no longer collapse to a single subunit.")
W("")
W("  Lookup summary:")
W(block(lookup_summary_df, max_rows=50))
W("")
W(f"  Total resolved component–target pairs: {len(binding_profiles_df)}")
W("")
W("-" * 100)
W("3. OPEN TARGETS WEIGHTS & STRING NETWORK")
W("-" * 100)
for d, w in disorder_weights.items():
    nz = sum(1 for v in w.values() if v > 0)
    top5 = sorted(w.items(), key=lambda x: x[1], reverse=True)[:5]
    W(f"  {d}: {nz}/{len(anchor_genes)} anchors scored; top5 = " +
      ", ".join(f"{g}:{s:.3f}" for g, s in top5))
W("")
W(f"  STRING graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges "
  f"(required_score >= {STRING_SCORE_CUTOFF}).")
W(f"  Anchors present in graph: {len(anchor_in_G)}/{len(anchor_genes)}.")
W("")
W("-" * 100)
W("4. CGR-LITE COMPONENT RANKING (mean unique_delta across 4 disorders)")
W("-" * 100)
W("  Higher mean unique_delta => component is more irreplaceable (its removal costs more coverage).")
W("")
W(block(cgr_lite_summary))
W("")
W("  Per-mode, per-disorder LOO tables are saved as:")
W("    results_loo_ablation_binding_only.csv")
W("    results_loo_ablation_with_metabolic.csv")
W("    results_loo_ablation_with_metabolic_and_pk.csv")
W("")
# PK-specific delta on the CYP2D6 inhibitor
try:
    pk_cyp = loo_pk_df[loo_pk_df["dropped"] == "Core_CYP2D6i"]["unique_delta"].mean()
    bo_cyp = loo_binding_df[loo_binding_df["dropped"] == "Core_CYP2D6i"]["unique_delta"].mean()
    W(f"  PK GATE EFFECT on Core_CYP2D6i (mean unique_delta): "
      f"binding_only={bo_cyp:.5f}  ->  with_metabolic_and_pk={pk_cyp:.5f}")
    W("  (The PK mode recovers the CYP2D6 inhibitor's functional role that static proximity misses.)")
except Exception:
    pass
W("")
W("-" * 100)
W("5. PSEUDO-pChEMBL SENSITIVITY (rank stability)")
W("-" * 100)
W(f"  Sweep values: {PSEUDO_PCHEMBL_GRID}. 'rank_range' = 0 means the component's rank is")
W("  invariant to the assumed curated affinity (robust); large rank_range = artifact risk.")
W("")
W(block(sens_pivot.reset_index()))
W("")
stable = sens_pivot[sens_pivot["rank_range"] == 0].index.tolist()
volatile = sens_pivot[sens_pivot["rank_range"] >= 2].index.tolist()
W(f"  Rank-invariant components : {stable if stable else 'none'}")
W(f"  Rank-volatile (>=2) comps : {volatile if volatile else 'none'}")
W("")
W("-" * 100)
W("6. DEGREE-MATCHED PERMUTATION NULL (empirical z / p)")
W("-" * 100)
W(f"  Each component's observed marginal delta is compared to N={N_PERM} random target sets of")
W("  equal size, drawn degree-bin-matched from the STRING graph. z>0 & p<0.05 => the component")
W("  contributes more coverage than expected from its target-set size/degree alone.")
W("")
W("  [with_metabolic]")
W(block(perm_full_df, floatfmt="{:.4f}"))
W("")
W("  [with_metabolic_and_pk]")
W(block(perm_pk_df, floatfmt="{:.4f}"))
W("")
try:
    sig = perm_full_df[(perm_full_df["p_value"] <= 0.05) & (perm_full_df["z_score"] > 0)]["component"].tolist()
    W(f"  Significant (p<=0.05, z>0, with_metabolic): {sig if sig else 'none'}")
except Exception:
    pass
W("")
W("-" * 100)
W("7. FDA (PHASE-4) SINGLE-MOLECULE MULTI-TARGET REPLACEMENT SCREEN")
W("-" * 100)
W(f"  Screen targets (CGR-target ∩ 121-anchor intersection, top by affinity): "
  f"{[t[0] for t in top_targets]}")
W("  Sourcing: ChEMBL mechanism.json (curated clinical links) + descending-sorted activity.json")
W(f"            (pChEMBL >= {FDA_ACT_PCHEMBL_MIN}); phase filter is type-safe float(max_phase)>=4.0.")
W(f"  Multi-target molecules examined: {len(multi_mcids)}")
W(f"  Phase-4 candidates found       : {len(fda_df)}")
W("")
W(block(fda_df, floatfmt="{:.4f}", max_rows=30))
W("")
W("-" * 100)
W("8. OUTPUT FILES")
W("-" * 100)
for p in sorted(Path("results").glob("*.csv")):
    W(f"  results/{p.name}")
W("  CGR_DETAILED_SUMMARY.txt  (this file)")
W("")
W("-" * 100)
W("9. CAVEATS")
W("-" * 100)
W("  • Curated mechanistic pChEMBL values are modelling assumptions, not measured affinities;")
W("    see Section 5 for their effect on ranks.")
W("  • STRING proximity is a static, undirected structural proxy; the PK mode (Section 4) is a")
W("    first-order correction for one known PK interaction (CYP2D6) only.")
W("  • Open Targets scores aggregate heterogeneous evidence; absolute magnitudes are relative.")
W("  • FDA 'replacement' candidates are network/affinity hypotheses, NOT clinical recommendations.")
W("=" * 100)

summary_text = "\n".join(L)
summary_path = Path("results/CGR_DETAILED_SUMMARY.txt")
summary_path.write_text(summary_text, encoding="utf-8")
print(summary_text[:4000])
print("\n... [summary truncated in console; full text written to file] ...")

# Append the summary to the existing zip (rebuild to guarantee inclusion).
zip_path = Path("cgr_network_pruning_results.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(Path("results").glob("*.csv")):
        zf.write(p, arcname=p.name)
    zf.write(summary_path, arcname=summary_path.name)
print(f"\n[OK] Wrote {summary_path} and rebundled everything into {zip_path}")

try:
    from google.colab import files
    files.download(str(zip_path))
except Exception as e:
    print(f"[INFO] Auto-download unavailable ({e}); download {zip_path} manually from the file browser.")

CGR NETWORK PRUNING · LOO ABLATION · FDA REPLACEMENT SCREEN — DETAILED RUN SUMMARY
Generated         : 2026-11-25 06:52:10 UTC
Pipeline version  : CGR-Network-Pruning/2.0 (publication-grade, cached, 3-mode)

----------------------------------------------------------------------------------------------------
0. METHODS OVERVIEW
----------------------------------------------------------------------------------------------------
This pipeline queries ChEMBL (bioactivity + curated mechanisms), Open Targets (genetic
association weights) and STRING (human PPI, score cutoff below) live, then runs an
affinity-weighted leave-one-out (LOO) ablation of the CGR stack across MDD, BIP, SCZ and
OCD. Component importance = mean 'unique_delta' = drop in disorder-weighted module
coverage when a component is removed. Coverage uses network proximity from each disorder
anchor to the nearest drug target, scaled by target affinity:

    eff_distance = shortest_path(anchor, target) / max(1, pChEMBL - 4.5)
   

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# The End